## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH05/ch05_LTX.ipynb)                                             

# About this Notebook

This notebook demonstrates **text-to-video** and **image-conditioned video generation** using the **LTX-Video** family of models developed by Lightricks. These models leverage efficient transformer-based architectures and support **quantized inference (8-bit)** for resource-optimized deployment on modern GPUs. The notebook uses both the standard `LTXPipeline` and `LTXConditionPipeline` for flexible video generation workflows.

### Steps Included:

1. **Conditional Video Generation (Image + Text)**:
   The notebook then demonstrates the use of `LTXConditionPipeline` for **image-conditioned video synthesis**. An image (e.g., of a woman walking on a tree-lined street) is downloaded and used as the visual anchor. This image, along with a descriptive prompt, is used to guide motion and scene synthesis. A **negative prompt** is added to suppress visual artifacts like blurriness or jitter.

2. **Fine-Grained Control Parameters**:

   * **Condition strength** allows for controlling how much the input image influences the generation.
   * **Image conditioning noise scale** introduces stochasticity while preserving structural alignment.
   * **Guidance scale** adjusts how strongly the model follows the text prompt.



# Installs

In [2]:
!pip install -q diffusers==0.50.1 gdown==6.1.0 bitsandbytes==0.39.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 66.9 MB/s eta 0:00:00


# Imports

In [1]:
import gdown
import torch
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig, LTXVideoTransformer3DModel, LTXPipeline
from diffusers.utils import export_to_video, load_video, load_image
from transformers import BitsAndBytesConfig as BitsAndBytesConfig, T5EncoderModel
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXConditionPipeline, LTXVideoCondition

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Get image for conditional video generation

In [2]:
file_id = "1UMBjsG2AqyGAtdaKCepjw97XOeWZqLVp"
gdown.download(id=file_id, output="image.jpg", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1UMBjsG2AqyGAtdaKCepjw97XOeWZqLVp
To: /content/image.jpg
100%|██████████| 4.66M/4.66M [00:00<00:00, 9.23MB/s]


'image.jpg'

# Load model and generate video

In [3]:
# Load pipeline
pipe = LTXConditionPipeline.from_pretrained(
    "Lightricks/LTX-Video-0.9.7-distilled",
    torch_dtype=torch.bfloat16
)
pipe.to("cuda")

# Load input image
image = load_image("image.jpg")

# Set conditioning
condition1 = LTXVideoCondition(
    image=image,
    frame_index=0,
    strength=1.0  # Full conditioning
)

# Set prompts
prompt = "A woman walking on a street with trees left and right. Above her, the vibrant green leaves shimmer in the dappled sunlight, their color almost fluorescent against the darker trunks. The air is alive with energy, sunlight flickering through the canopy, painting shifting patterns across the trees."
negative_prompt = "worst quality, inconsistent motion, blurry, jittery, distorted"

# Set reproducible generator
generator = torch.Generator("cuda").manual_seed(42)

# Run generation
video = pipe(
    conditions=[condition1],
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=704,
    height=512,
    num_frames=48,  # 2 seconds at 24 FPS
    frame_rate=24,
    num_inference_steps=40,
    guidance_scale=5.5,
    image_cond_noise_scale=0.15,
    generator=generator
).frames[0]

# Export video
export_to_video(video, "walking.mp4", fps=24)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


model_index.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

'walking.mp4'

In [4]:
from IPython.display import Video

Video("walking.mp4", embed=True)